In [ ]:
!pip install -q faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 17.5 MB/s eta 0:00:00


In [ ]:
import json
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE  = "/content/drive/MyDrive/arxiv-llm-project"   # adjust to your shared folder
INDEX_FILE  = f"{DRIVE_BASE}/data/processed/faiss_index.bin"
META_FILE   = f"{DRIVE_BASE}/data/processed/chunk_metadata.jsonl"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

Mounted at /content/drive


In [ ]:
print("Loading FAISS index...")
index = faiss.read_index(INDEX_FILE)
print(f"  Vectors in index: {index.ntotal:,}")

print("Loading chunk metadata...")
metadata = []
with open(META_FILE, "r") as f:
    for line in f:
        metadata.append(json.loads(line.strip()))

assert len(metadata) == index.ntotal, \
    f"MISMATCH: {len(metadata)} metadata records vs {index.ntotal} index vectors"
print(f"  Metadata records: {len(metadata):,} ✓")

Loading FAISS index...
  Vectors in index: 76,443
Loading chunk metadata...
  Metadata records: 76,443 ✓


In [ ]:
print("\nLoading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)
print("  Ready ✓")


Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  Ready ✓


In [ ]:
def retrieve(query: str, k: int = 3) -> list[dict]:
    """
    Embed a query and return top-k most relevant chunks.

    Args:
        query: free-form research question or topic
        k:     number of chunks to return (use 3 for ~1500 token context budget,
               5 if model handles longer prompts without degrading)

    Returns:
        list of dicts with keys: rank, distance, chunk_id, paper_id,
                                  title, url, year, text_preview
    """
    q_emb = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    distances, indices = index.search(q_emb, k)

    results = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        meta = metadata[idx]
        results.append({
            "rank":         rank + 1,
            "distance":     float(dist),
            "chunk_id":     meta.get("chunk_id", str(idx)),
            "paper_id":     meta["paper_id"],
            "title":        meta.get("title", "N/A"),
            "url":          meta.get("url", ""),
            "year":         meta.get("year", ""),
            "text_preview": meta.get("text_preview", "")[:500],
        })
    return results

In [ ]:
def build_rag_prompt(query: str, chunks: list[dict]) -> str:
    """
    Construct a RAG prompt from retrieved chunks.
    Keeps each chunk to 400 chars to stay within context window at k=3.
    """
    context_parts = []
    for c in chunks:
        context_parts.append(
            f"[Source {c['rank']}: {c['title']} ({c['year']})]\n"
            f"{c['text_preview'][:400]}"
        )
    context_block = "\n\n".join(context_parts)

    prompt = (
        f"You are a research assistant specializing in machine learning.\n"
        f"Use the following retrieved paper excerpts to answer the question.\n"
        f"If the answer is not in the context, say so.\n\n"
        f"CONTEXT:\n{context_block}\n\n"
        f"Question: {query}\n"
        f"Answer:"
    )
    return prompt

In [ ]:
TEST_QUERIES = [
    # Core ML topics
    "What are the main advantages of transformer architectures over RNNs?",
    "How does contrastive learning work in self-supervised settings?",
    "What methods exist for reducing hallucinations in large language models?",
    # Applied / domain-specific
    "Graph neural networks for molecular property prediction",
    "Efficient fine-tuning methods for large language models",
    "Reinforcement learning from human feedback RLHF alignment",
    # Methodology
    "How is knowledge distillation used to compress neural networks?",
    "What is the role of data augmentation in vision transformers?",
    # Broader
    "Federated learning privacy preserving machine learning",
    "Diffusion models image generation score matching",
]

print("=" * 65)
print("RAG RETRIEVAL — 10 SAMPLE QUERIES")
print("=" * 65)

for q in TEST_QUERIES:
    print(f"\nQuery: {q}")
    print("-" * 55)
    hits = retrieve(q, k=3)
    for h in hits:
        print(f"  [{h['rank']}] dist={h['distance']:.4f} | {h['title'][:55]}")
        print(f"       {h['text_preview'][:120]}...")
    print()

RAG RETRIEVAL — 10 SAMPLE QUERIES

Query: What are the main advantages of transformer architectures over RNNs?
-------------------------------------------------------
  [1] dist=0.7050 | Anatomy of Neural Language Models
       reconstructed [12, 62]. For this reason, the factorized probability represents an approximation of the conditional 
prob...
  [2] dist=0.7306 | Transferability in Deep Learning: A Survey
       assumption in Convolutional Neural Network (CNN) and Recurrent Neural Network (RNN).
A strong inductive bias makes pre-t...
  [3] dist=0.7412 | Multimodal Deep Learning
       2.1 State-of-the-art in NLP
21

et al. (2022)).
data sets. indicates the minimum number of steps before the respective s...


Query: How does contrastive learning work in self-supervised settings?
-------------------------------------------------------
  [1] dist=0.6598 | Preserving Silent Features for Domain Generalization
       (a) Supervised pre-trained model
(b) Self-supervised contrastive pre-

In [ ]:
COMPARISON_QUERY = "attention mechanism self-supervised learning vision"

print("=" * 65)
print(f"k=3 vs k=5 COMPARISON")
print(f"Query: {COMPARISON_QUERY}")
print("=" * 65)

for k in [3, 5]:
    hits   = retrieve(COMPARISON_QUERY, k=k)
    prompt = build_rag_prompt(COMPARISON_QUERY, hits)
    print(f"\n── k={k} ──────────────────────────────────────────────")
    print(f"  Prompt length (chars): {len(prompt):,}")
    print(f"  Approx tokens:         {len(prompt) // 4:,}   (budget: ~2048)")
    for h in hits:
        print(f"  [{h['rank']}] {h['title'][:60]}")

print("\n→ Pick k=3 if approx tokens > 1800 with k=5")
print("→ Pick k=5 if both fit comfortably under 2000 tokens")


# ── Cell 9: Show a full RAG prompt example ───────────────────
print("\n" + "=" * 65)
print("FULL RAG PROMPT EXAMPLE (k=3)")
print("=" * 65)

sample_query = "What techniques reduce catastrophic forgetting in continual learning?"
sample_hits  = retrieve(sample_query, k=3)
sample_prompt = build_rag_prompt(sample_query, sample_hits)
print(sample_prompt)
print(f"\nTotal prompt length: {len(sample_prompt):,} chars / ~{len(sample_prompt)//4:,} tokens")

k=3 vs k=5 COMPARISON
Query: attention mechanism self-supervised learning vision

── k=3 ──────────────────────────────────────────────
  Prompt length (chars): 1,754
  Approx tokens:         438   (budget: ~2048)
  [1] GTA: Guided Transfer of Spatial Attention from Object-Centri
  [2] Synthesizer Based Efficient Self-Attention for Vision Tasks
  [3] Real-World Graph Convolution Networks (RW-GCNs) for Action R

── k=5 ──────────────────────────────────────────────
  Prompt length (chars): 2,710
  Approx tokens:         677   (budget: ~2048)
  [1] GTA: Guided Transfer of Spatial Attention from Object-Centri
  [2] Synthesizer Based Efficient Self-Attention for Vision Tasks
  [3] Real-World Graph Convolution Networks (RW-GCNs) for Action R
  [4] Transferring Pre-trained Multimodal Representations with Cro
  [5] Multimodal Deep Learning

→ Pick k=3 if approx tokens > 1800 with k=5
→ Pick k=5 if both fit comfortably under 2000 tokens

FULL RAG PROMPT EXAMPLE (k=3)
You are a research assista

In [ ]:
MODULE_PATH = f"{DRIVE_BASE}/app/rag_pipeline.py"
os.makedirs(f"{DRIVE_BASE}/app", exist_ok=True)

MODULE_CODE = '''"""
rag_pipeline.py — shared RAG retrieval module
Built in Milestone 3.1, imported by M3 in Milestone 3.2
"""

import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

class RAGPipeline:
    def __init__(self, index_path: str, metadata_path: str, k: int = 3):
        print("Loading FAISS index...")
        self.index = faiss.read_index(index_path)
        print(f"  {self.index.ntotal:,} vectors loaded")

        print("Loading metadata...")
        self.metadata = []
        with open(metadata_path, "r") as f:
            for line in f:
                self.metadata.append(json.loads(line.strip()))

        print("Loading embedder...")
        self.embedder = SentenceTransformer(EMBED_MODEL)
        self.k = k
        print("RAGPipeline ready ✓")

    def retrieve(self, query: str, k: int = None) -> list[dict]:
        k = k or self.k
        q_emb = self.embedder.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype(np.float32)

        distances, indices = self.index.search(q_emb, k)

        results = []
        for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
            meta = self.metadata[idx]
            results.append({
                "rank":         rank + 1,
                "distance":     float(dist),
                "chunk_id":     meta.get("chunk_id", str(idx)),
                "paper_id":     meta["paper_id"],
                "title":        meta.get("title", "N/A"),
                "url":          meta.get("url", ""),
                "year":         meta.get("year", ""),
                "text_preview": meta.get("text_preview", "")[:500],
            })
        return results

    def build_prompt(self, query: str, chunks: list[dict]) -> str:
        context_parts = []
        for c in chunks:
            context_parts.append(
                f"[Source {c[\'rank\']}: {c[\'title\']} ({c[\'year\']})]\n"
                f"{c[\'text_preview\'][:400]}"
            )
        context_block = "\\n\\n".join(context_parts)

        return (
            f"You are a research assistant specializing in machine learning.\\n"
            f"Use the following retrieved paper excerpts to answer the question.\\n"
            f"If the answer is not in the context, say so.\\n\\n"
            f"CONTEXT:\\n{context_block}\\n\\n"
            f"Question: {query}\\nAnswer:"
        )

    def query(self, question: str, k: int = None) -> tuple[str, list[dict]]:
        """Convenience method: retrieve + build prompt in one call."""
        chunks = self.retrieve(question, k=k)
        prompt = self.build_prompt(question, chunks)
        return prompt, chunks
'''

with open(MODULE_PATH, "w") as f:
    f.write(MODULE_CODE)

print(f"\n✅ rag_pipeline.py saved → {MODULE_PATH}")
print("   M3 can import this in 3.2 with:")
print("   from app.rag_pipeline import RAGPipeline")
print("   rag = RAGPipeline(index_path=..., metadata_path=...)")

print("\n✅ Milestone 3.1 complete")
print("   Next: 3.2 — M2 wraps generate(), M3 wires RAGPipeline into Gradio")


✅ rag_pipeline.py saved → /content/drive/MyDrive/arxiv-llm-project/app/rag_pipeline.py
   M3 can import this in 3.2 with:
   from app.rag_pipeline import RAGPipeline
   rag = RAGPipeline(index_path=..., metadata_path=...)

✅ Milestone 3.1 complete
   Next: 3.2 — M2 wraps generate(), M3 wires RAGPipeline into Gradio
